In [1]:
import functools
import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from tqdm.notebook import trange, tqdm
from torchvision.datasets import FashionMNIST
from torch.optim import Adam
from torch.utils.data import DataLoader

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True


References:

- [[Song21]](https://openreview.net/pdf?id=PxTIG12RRHS) Score-Based Generative Modeling Through Stochastic Differential Equations
- Yang Song's blog

## Variance Preserving SDE (VP)

Variance Preserving diffusion SDE (Eq 11 in [Song21]):

\begin{align*}
d \mathbf{x} = - \frac{1}{2} \beta(t) \mathbf{x} dt + \sqrt{\beta(t)}d\mathbf{w}
\end{align*}

This follows the general SDE form $d \mathbf{x} = f(\mathbf{x}, t) dt + g(t) d \mathbf{w}$. We call $f(\mathbf{x}, t)$ the **drift coefficient** and $g(t)$ the **diffusion coefficient**.

The corresponding general conditional linear Gaussian distribution is:
\begin{align*}
p(x_t|x_0) = \mathcal{N}(x_t; \alpha(t)x_0, \sigma^2(t)I)
\end{align*}
where $\alpha: [0,1] \rightarrow \mathbb{R}$,  $\sigma: [0,1] \rightarrow \mathbb{R}$

$\mu(t), \sigma(t)$ can be derived analytically from $f(\mathbf{x}, t), g(t)$.

\begin{align*}
\begin{cases}
  \mu(t) = \alpha(t)x_0 = \exp{(c(t))}x_0 \\
  \sigma^2(t) = 1 - \exp(2c(t))
\end{cases}
\end{align*}

### Implementing the VP SDE

Refer to Equations (32) and (33) in [Song21] to identify:
* $\beta(t)$
* $c(t)$
* $\mu(t)$
* $\sigma(t)$

Copy your code from `vp.py`

In [17]:
"""
diffusion/vp.py  —  Variance-Preserving (VP) SDE
=================================================
Part 5 of EE/CS 148B HW4.

Reference: Song et al. (2021) "Score-Based Generative Modeling through
Stochastic Differential Equations" (Song21), Appendix B & D.

Students implement every method marked TODO.  Methods marked PROVIDED
are complete and should not be modified.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
from torch import Tensor


def _broadcast_time(t: Tensor, x: Tensor) -> Tensor:
    """Reshape (B,) time to broadcast with (B, *spatial)."""
    return t.view(t.shape[0], *([1] * (x.dim() - 1)))


class VPSDE:
    """Variance-Preserving SDE forward process and samplers.

    The VP-SDE is:
        dx = -½ β(t) x dt + √β(t) dB_t

    with β(t) = β_min + (β_max - β_min) * t  (linear schedule).

    Args:
        beta_min: Minimum noise schedule value β_min.
        beta_max: Maximum noise schedule value β_max.
        T:        Number of discrete time steps (used by the EM/PC samplers).
    """

    def __init__(self, beta_min: float = 0.01, beta_max: float = 5.0, T: int = 1000):
        self.beta_min = beta_min
        self.beta_max = beta_max
        self.T = T

        self._register_alphas()

    # ------------------------------------------------------------------
    # 5.A  Defining the VP SDE
    # ------------------------------------------------------------------

    def beta(self, t: Tensor) -> Tensor:
        """β(t) — the linear noise schedule.

        Args:
            t: Continuous time in [0, 1], shape (*).

        Returns:
            β(t), same shape as t.

        Reference: Eq. (32) of Song21.
        """
        return self.beta_min + (self.beta_max - self.beta_min) * t

    def c(self, t: Tensor) -> Tensor:
        """c(t) = exp(-½ ∫_0^t β(s) ds) — the signal decay factor.

        For a linear β schedule:
            ∫_0^t β(s) ds = β_min * t + ½ (β_max - β_min) * t²

        Args:
            t: Continuous time in [0, 1], shape (*).

        Returns:
            c(t), same shape as t.

        Reference: Eq. (33) of Song21.
        """
        integral = self.beta_min * t + 0.5 * (self.beta_max - self.beta_min) * t ** 2
        return torch.exp(-0.5 * integral)

    def sigma(self, t: Tensor) -> Tensor:
        """σ(t) = √(1 - c(t)²) — the noise standard deviation.

        Args:
            t: Continuous time in [0, 1], shape (*).

        Returns:
            σ(t), same shape as t.
        """
        return torch.sqrt(1.0 - self.c(t) ** 2)

    def drift(self, x: Tensor, t: Tensor) -> Tensor:
        """Drift coefficient  f(x, t) = -½ β(t) x.

        Args:
            x: State tensor, shape (B, *).
            t: Time tensor, shape (B,) broadcast-compatible with x.

        Returns:
            Drift f(x, t), same shape as x.
        """
        b = self.beta(t)
        return -0.5 * _broadcast_time(b, x) * x

    def diffusion(self, t: Tensor) -> Tensor:
        """Diffusion coefficient  g(t) = √β(t).

        Args:
            t: Time tensor, shape (*).

        Returns:
            g(t), same shape as t.
        """
        return torch.sqrt(self.beta(t))

    def marginal(self, x0: Tensor, t: Tensor) -> tuple[Tensor, Tensor]:
        """Sample from the forward marginal  q(x_t | x_0).

        The marginal satisfies:
            x_t = c(t) * x_0 + σ(t) * ε,   ε ~ N(0, I)

        Args:
            x0: Clean data, shape (B, *).
            t:  Continuous time in [0, 1], shape (B,).

        Returns:
            (x_t, eps): noised sample and the noise used, both shape (B, *).
        """
        eps = torch.randn_like(x0)
        c_t = _broadcast_time(self.c(t), x0)
        s_t = _broadcast_time(self.sigma(t), x0)
        x_t = c_t * x0 + s_t * eps
        return x_t, eps


    @property
    def num_steps(self) -> int:
        """Alias for T (used by notebook samplers)."""
        return self.T

    def f(self, x: Tensor, t: Tensor) -> Tensor:
        """Forward drift f(x, t) = -½ β(t) x (notebook alias)."""
        return self.drift(x, t)

    def g(self, t: Tensor) -> Tensor:
        """Diffusion g(t) = √β(t) (notebook alias)."""
        return self.diffusion(t)

    def _register_alphas(self) -> None:
        """Discrete α(t) = c(t)² for PC corrector steps."""
        t = torch.linspace(0.0, 1.0, self.T)
        integral = self.beta_min * t + 0.5 * (self.beta_max - self.beta_min) * t ** 2
        c = torch.exp(-0.5 * integral)
        self.alphas = c ** 2

    # ------------------------------------------------------------------
    # 5.B  Samplers
    # ------------------------------------------------------------------

    def _reverse_drift(
        self, x: Tensor, t: Tensor, score: Tensor
    ) -> tuple[Tensor, Tensor]:
        """Reverse-SDE drift and diffusion at time t."""
        b = self.beta(t)
        b_bc = _broadcast_time(b, x)
        drift = -0.5 * b_bc * x - b_bc * score
        diff = torch.sqrt(b_bc)
        return drift, diff

    @torch.no_grad()
    def euler_maruyama(
        self,
        score_model: nn.Module,
        shape: tuple[int, ...],
        num_steps: int | None = None,
        device: str | torch.device = "cpu",
    ) -> Tensor:
        """Euler-Maruyama reverse-SDE sampler (Problem 5.B.i).

        Starting from x(T=1) ~ N(0, σ(1)² I), integrates the reverse VP-SDE:
            dx = [-½ β(t) x - β(t) ∇_x log p_t(x)] dt + √β(t) dB̄_t

        Args:
            score_model: Trained score network s_θ(x, t).
                         Called as `score_model(x, t)` where t is a float
                         tensor of shape (B,) with values in [0, 1].
            shape:       Output shape (B, C, H, W).
            num_steps:   Number of discretisation steps (default: self.T).
            device:      Target device.

        Returns:
            Generated samples, shape (B, C, H, W), values in [-1, 1].
        """
        num_steps = num_steps or self.T
        device = torch.device(device)
        B = shape[0]
        delta_t = 1.0 / num_steps
        dt = -delta_t

        # x(t=1) = z * σ(t=1),  z ~ N(0, I);  σ(t) = √(1 - c(t)²)
        t = torch.ones(B, device=device)
        z = torch.randn(shape, device=device)
        sigma_t = self.sigma(t).view(B, *([1] * (len(shape) - 1)))
        x = z * sigma_t

        for i in range(num_steps):
            t = torch.full((B,), 1.0 - i * delta_t, device=device)
            score = score_model(x, t)
            beta_t = self.beta(t).view(B, *([1] * (len(shape) - 1)))
            rev_drift = -0.5 * beta_t * x - beta_t * score
            xi = torch.randn_like(x)
            x = x + rev_drift * dt + torch.sqrt(beta_t) * math.sqrt(delta_t) * xi

        return x.clamp(0.0, 1.0)

    def _langevin_corrector(
        self,
        x: Tensor,
        t: Tensor,
        score_model: nn.Module,
        n_corrector: int,
        snr: float,
    ) -> Tensor:
        """Annealed Langevin corrector steps (Algorithm 5, Song21)."""
        for _ in range(n_corrector):
            score = score_model(x, t)
            noise = torch.randn_like(x)
            grad_norm = torch.norm(score.reshape(score.shape[0], -1), dim=-1).mean()
            noise_norm = torch.norm(noise.reshape(noise.shape[0], -1), dim=-1).mean()
            alpha = self.c(t) ** 2
            step_size = (snr * noise_norm / (grad_norm + 1e-8)) ** 2 * 2.0 * alpha
            step_bc = _broadcast_time(step_size, x)
            x = x + step_bc * score + torch.sqrt(2.0 * step_bc) * noise
        return x

    @torch.no_grad()
    def predictor_corrector(
        self,
        score_model: nn.Module,
        shape: tuple[int, ...],
        num_steps: int | None = None,
        n_corrector: int = 1,
        snr: float = 0.16,
        device: str | torch.device = "cpu",
    ) -> Tensor:
        """Predictor-Corrector sampler with EM predictor (Problem 5.B.ii).

        Follows Algorithm 5 of Song21.  Each predictor step is an EM step;
        each corrector step is one step of annealed Langevin dynamics.

        Args:
            score_model:  Trained score network s_θ(x, t).
            shape:        Output shape (B, C, H, W).
            num_steps:    Number of predictor steps (default: self.T).
            n_corrector:  Number of Langevin corrector steps per predictor step.
            snr:          Signal-to-noise ratio for the corrector step size.
            device:       Target device.

        Returns:
            Generated samples, shape (B, C, H, W), values in [-1, 1].
        """
        num_steps = num_steps or self.T
        device = torch.device(device)
        B = shape[0]
        dt = -1.0 / num_steps

        t1 = torch.ones(B, device=device)
        s1 = self.sigma(t1).view(B, *([1] * (len(shape) - 1)))
        x = torch.randn(shape, device=device) * s1

        for i in range(num_steps):
            t_val = 1.0 - i / num_steps
            t = torch.full((B,), t_val, device=device)

            x = self._langevin_corrector(x, t, score_model, n_corrector, snr)

            score = score_model(x, t)
            drift, diff = self._reverse_drift(x, t, score)
            noise = torch.randn_like(x)
            x = x + drift * dt + diff * math.sqrt(-dt) * noise

        return x.clamp(0.0, 1.0)

    # ------------------------------------------------------------------
    # 5.D  Inverse problems (EC)
    # ------------------------------------------------------------------

    @torch.no_grad()
    def inpaint(
        self,
        score_model: nn.Module,
        corrupted: Tensor,
        mask: Tensor,
        num_steps: int | None = None,
        device: str | torch.device = "cpu",
    ) -> Tensor:
        """Conditional reverse diffusion for inpainting (EC Problem 5.D).

        At each reverse step, replaces the known pixels with their
        forward-diffused ground-truth values, conditioning the reverse
        process on the observed measurements.

        Reference: Song et al. (2022) "Solving Inverse Problems in Medical
        Imaging with Score-Based Generative Models".

        Args:
            score_model: Trained score network s_θ(x, t).
            corrupted:   Observed (corrupted) image, shape (B, C, H, W).
                         Unknown pixels are set to 0.
            mask:        Binary mask, shape (B, 1, H, W).
                         1 = observed pixel, 0 = missing pixel.
            num_steps:   Reverse steps (default: self.T).
            device:      Target device.

        Returns:
            Reconstructed images, shape (B, C, H, W).
        """
        num_steps = num_steps or self.T
        device = torch.device(device)
        shape = corrupted.shape
        B = shape[0]
        dt = -1.0 / num_steps
        corrupted = corrupted.to(device)
        mask = mask.to(device)

        t1 = torch.ones(B, device=device)
        s1 = self.sigma(t1).view(B, *([1] * (len(shape) - 1)))
        x = torch.randn(shape, device=device) * s1

        for i in range(num_steps):
            t_val = 1.0 - i / num_steps
            t = torch.full((B,), t_val, device=device)

            score = score_model(x, t)
            drift, diff = self._reverse_drift(x, t, score)
            noise = torch.randn_like(x)
            x = x + drift * dt + diff * math.sqrt(-dt) * noise

            c_t = _broadcast_time(self.c(t), corrupted)
            s_t = _broadcast_time(self.sigma(t), corrupted)
            eps = torch.randn_like(corrupted)
            x_known = c_t * corrupted + s_t * eps
            x = x * (1.0 - mask) + x_known * mask

        return x.clamp(0.0, 1.0)
    def marginal_proba(self, x, t):
        """
        Return the mean and std of q(x_t | x_0):
            x_t = c(t) x_0 + sigma(t) eps
        """
        mean = self.c(t)[:, None, None, None] * x
        std = self.sigma(t)
        return mean, std


## Sampling with VP sde

As per Appendix E of [Song21], recall that for any SDE of the form
\begin{align*}
d \mathbf{x} = \mathbf{f}(\mathbf{x}, t) dt + g(t) d\mathbf{w},
\end{align*}
the reverse-time SDE is given by
\begin{align*}
d \mathbf{x} = [\mathbf{f}(\mathbf{x}, t) - g(t)^2 \nabla_\mathbf{x} \log p_t(\mathbf{x})] dt + g(t) d \bar{\mathbf{w}}.
\end{align*}

We use the [Euler-Maruyama](https://en.wikipedia.org/wiki/Euler%E2%80%93Maruyama_method) numerical method to solve for the reverse-time SDE. This method relies on discretizing the SDE, replacing $dt$ with $\Delta t$ and $d \mathbf{w}$ with $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, g^2(t) \Delta t \mathbf{I})$.

This lead to the following iteration rule:
\begin{align}
\mathbf{x}_{t-\Delta t} =
\mathbf{x}_t
- \mathbf{f}(\mathbf{x}_t, t)\Delta t
+ g^2(t) s_\theta(\mathbf{x}_t, t)\Delta t
+ g(t)\sqrt{\Delta_t}\mathbf{z}_t.
\end{align}

Note: for the last step (i.e. $t-\Delta_t = 0$), we do not wish to add back noise ($g(t)\sqrt{\Delta_t}\mathbf{z}_t$).

In [4]:
# Problem 5.C.iii — 64 EM samples
score_model.eval()
em_samples = Euler_Maruyama_sampler(score_model, sde, n_samples, num_steps=num_steps, device=device)
plot_images(em_samples.clamp(0, 1))
plt.title(f'EM samples (N={num_steps}, β=[{beta_min}, {beta_max}])')
plt.savefig(os.path.join(checkpoint_dir, f'em_samples_{beta_min}_{beta_max}.png'), dpi=150, bbox_inches='tight')
plt.show()


In [5]:
# Problem 5.C.iv — PC sampler, 1 corrector step
pc_samples_1 = predictor_corrector_sampler(
    score_model, sde, n_samples, num_steps=num_steps, num_corrector_steps=1, device=device,
)
plot_images(pc_samples_1.clamp(0, 1))
plt.title(f'PC samples — 1 corrector step')
plt.savefig(os.path.join(checkpoint_dir, f'pc_samples_n1_{beta_min}_{beta_max}.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Problem 5.C.iv — PC sampler, 3 corrector steps
pc_samples_3 = predictor_corrector_sampler(
    score_model, sde, n_samples, num_steps=num_steps, num_corrector_steps=3, device=device,
)
plot_images(pc_samples_3.clamp(0, 1))
plt.title('PC samples — 3 corrector steps')
plt.savefig(os.path.join(checkpoint_dir, f'pc_samples_n3_{beta_min}_{beta_max}.png'), dpi=150, bbox_inches='tight')
plt.show()



## Setup -- no TODOs to fill in here :)

### Config

In [6]:
# --- Part 5 hyperparameters (Problem 5.C) ---
n_epochs = 50
batch_size = 64          # training batch size
n_samples = 64           # number of images to plot / generate
lr = 1e-4
num_steps = 1000         # reverse SDE discretization steps
checkpoint_dir = './checkpoints/'

# VP noise schedule — primary run; try [0.01, 10] as an ablation
beta_min = 0.01
beta_max = 5.0
sde_params = [beta_min, beta_max]

# Early stopping: stop if validation loss does not improve for `patience` epochs
patience = 10
min_delta = 1e-4         # minimum val-loss decrease to count as improvement


### Dataset

In [7]:
train_transforms = transforms.Compose([transforms.ToTensor()])
train_dataset = FashionMNIST('.', train=True, transform=train_transforms, download=True)
val_dataset = FashionMNIST('.', train=False, transform=train_transforms, download=True)

num_workers = 2 if device.type == 'cuda' else 0
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=device.type=='cuda')
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=device.type=='cuda')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')


100%|██████████| 26.4M/26.4M [00:02<00:00, 10.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 209kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.92MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.7MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [11]:
from torchvision.utils import make_grid
def plot_images(images):
    sample_grid = make_grid(images, nrow=int(np.sqrt(images.shape[0])))
    plt.figure(figsize=(6,6))
    plt.axis('off')
    plt.imshow(sample_grid.cpu().permute(1, 2, 0).squeeze())
    plt.show()

### Problem 5.C.i — Visualize the data prior

Plot 64 FashionMNIST training images.


In [ ]:
os.makedirs(checkpoint_dir, exist_ok=True)
data_iter = iter(train_loader)
images, _ = next(data_iter)
images = images[:n_samples].to(device)
plot_images(images)
plt.title(f'FashionMNIST prior — {n_samples} training examples')
plt.savefig(os.path.join(checkpoint_dir, 'fashionmnist_prior_64.png'), dpi=150, bbox_inches='tight')
plt.show()


### Score-matching model

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class GaussianFourierProjection(nn.Module):
    """Gaussian random features for encoding time steps."""
    def __init__(self, embed_dim, scale=30.):
        super().__init__()
        # Randomly sample weights during initialization. These weights are fixed
        # during optimization and are not trainable.
        self.W = nn.Parameter(torch.randn(embed_dim // 2) * scale, requires_grad=False)
    def forward(self, x):
        x_proj = x[:, None] * self.W[None, :] * 2 * np.pi
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class Dense(nn.Module):
    """A fully connected layer that reshapes outputs to feature maps."""
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.dense = nn.Linear(input_dim, output_dim)
    def forward(self, x):
        return self.dense(x)[..., None, None]


class ScoreNet(nn.Module):
    """A time-dependent score-based model built upon U-Net architecture."""

    def __init__(self, marginal_proba, channels=[32, 64, 128, 256], embed_dim=256):
        """Initialize a time-dependent score-based network.

        Args:
          marginal_proba: A function that takes time t and gives the standard
            deviation of the perturbation kernel p_{0t}(x(t) | x(0)).
          channels: The number of channels for feature maps of each resolution.
          embed_dim: The dimensionality of Gaussian random feature embeddings.
        """
        super().__init__()
        # Gaussian random feature embedding layer for time
        self.embed = nn.Sequential(GaussianFourierProjection(embed_dim=embed_dim),
             nn.Linear(embed_dim, embed_dim))
        # Encoding layers where the resolution decreases
        self.conv1 = nn.Conv2d(1, channels[0], 3, stride=1, bias=False)
        self.dense1 = Dense(embed_dim, channels[0])
        self.gnorm1 = nn.GroupNorm(4, num_channels=channels[0])
        self.conv2 = nn.Conv2d(channels[0], channels[1], 3, stride=2, bias=False)
        self.dense2 = Dense(embed_dim, channels[1])
        self.gnorm2 = nn.GroupNorm(32, num_channels=channels[1])
        self.conv3 = nn.Conv2d(channels[1], channels[2], 3, stride=2, bias=False)
        self.dense3 = Dense(embed_dim, channels[2])
        self.gnorm3 = nn.GroupNorm(32, num_channels=channels[2])
        self.conv4 = nn.Conv2d(channels[2], channels[3], 3, stride=2, bias=False)
        self.dense4 = Dense(embed_dim, channels[3])
        self.gnorm4 = nn.GroupNorm(32, num_channels=channels[3])

        # Decoding layers where the resolution increases
        self.tconv4 = nn.ConvTranspose2d(channels[3], channels[2], 3, stride=2, bias=False)
        self.dense5 = Dense(embed_dim, channels[2])
        self.tgnorm4 = nn.GroupNorm(32, num_channels=channels[2])
        self.tconv3 = nn.ConvTranspose2d(channels[2] + channels[2], channels[1], 3, stride=2, bias=False, output_padding=1)
        self.dense6 = Dense(embed_dim, channels[1])
        self.tgnorm3 = nn.GroupNorm(32, num_channels=channels[1])
        self.tconv2 = nn.ConvTranspose2d(channels[1] + channels[1], channels[0], 3, stride=2, bias=False, output_padding=1)
        self.dense7 = Dense(embed_dim, channels[0])
        self.tgnorm2 = nn.GroupNorm(32, num_channels=channels[0])
        self.tconv1 = nn.ConvTranspose2d(channels[0] + channels[0], 1, 3, stride=1)

        # The swish activation function
        self.act = lambda x: x * torch.sigmoid(x)
        self.marginal_proba = marginal_proba

    def forward(self, x, t):
        # Obtain the Gaussian random feature embedding for t
        embed = self.act(self.embed(t))
        # Encoding path
        h1 = self.conv1(x)
        ## Incorporate information from t
        h1 += self.dense1(embed)
        ## Group normalization
        h1 = self.gnorm1(h1)
        h1 = self.act(h1)
        h2 = self.conv2(h1)
        h2 += self.dense2(embed)
        h2 = self.gnorm2(h2)
        h2 = self.act(h2)
        h3 = self.conv3(h2)
        h3 += self.dense3(embed)
        h3 = self.gnorm3(h3)
        h3 = self.act(h3)
        h4 = self.conv4(h3)
        h4 += self.dense4(embed)
        h4 = self.gnorm4(h4)
        h4 = self.act(h4)

        # Decoding path
        h = self.tconv4(h4)
        ## Skip connection from the encoding path
        h += self.dense5(embed)
        h = self.tgnorm4(h)
        h = self.act(h)
        h = self.tconv3(torch.cat([h, h3], dim=1))
        h += self.dense6(embed)
        h = self.tgnorm3(h)
        h = self.act(h)
        h = self.tconv2(torch.cat([h, h2], dim=1))
        h += self.dense7(embed)
        h = self.tgnorm2(h)
        h = self.act(h)
        h = self.tconv1(torch.cat([h, h1], dim=1))

        # Normalize output
        _, std = self.marginal_proba(x, t)
        h = h / std[:, None, None, None]
        return h

### Loss function

In [9]:
def loss_fn(model, x, sde, eps=1e-5):
    """ Inputs:
          model: score model (i.e. diffusion model)
          x: batch of images
          sde: instance of VP class
          eps: parameter for numerical stability (1e-5 for learning, 1e-3 for sampling)
    """
    random_t = torch.rand(x.shape[0], device=x.device) * (1. - eps) + eps
    z = torch.randn_like(x, device=x.device)
    mean, std = sde.marginal_proba(x, random_t)
    perturbed_x = mean + z * std[:, None, None, None]

    # predict the score function for each perturbed x in the batch and its corresponding random t
    score = model(perturbed_x, random_t)

    # compute loss
    losses = score * std[:, None, None, None] + z
    loss = torch.mean(torch.sum(losses**2, dim=(1,2,3)))
    return loss

## Train the score model

In [15]:
def train(sde_params, force_retrain=False):
    """Train VP score model with validation, checkpointing, and early stopping."""
    beta_min, beta_max = sde_params
    sde = VPSDE(beta_min, beta_max, num_steps)

    score_model = ScoreNet(marginal_proba=sde.marginal_proba).to(device)
    optimizer = Adam(score_model.parameters(), lr=lr)

    os.makedirs(checkpoint_dir, exist_ok=True)
    params_str = f'{beta_min}_{beta_max}'
    checkpoint_path = os.path.join(
        checkpoint_dir, f'ckpt_fashionmnist_{n_epochs}epochs_{params_str}.pth'
    )
    loss_path = os.path.join(checkpoint_dir, f'losses_{params_str}.npz')

    train_losses, val_losses = [], []
    if os.path.exists(loss_path) and not force_retrain:
        data = np.load(loss_path)
        train_losses = data['train'].tolist()
        val_losses = data['val'].tolist()

    if os.path.exists(checkpoint_path) and not force_retrain:
        score_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f'Loaded checkpoint: {checkpoint_path}')
        return score_model, sde, train_losses, val_losses

    best_val = float('inf')
    patience_counter = 0

    for epoch in trange(n_epochs, desc='Training'):
        score_model.train()
        running, n = 0.0, 0
        for x, _ in train_loader:
            x = x.to(device)
            loss = loss_fn(score_model, x, sde)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item() * x.size(0)
            n += x.size(0)
        train_loss = running / n
        train_losses.append(train_loss)

        score_model.eval()
        running, n = 0.0, 0
        with torch.no_grad():
            for x, _ in val_loader:
                x = x.to(device)
                running += loss_fn(score_model, x, sde).item() * x.size(0)
                n += x.size(0)
        val_loss = running / n
        val_losses.append(val_loss)

        print(f'Epoch {epoch+1:3d}/{n_epochs} | train {train_loss:.4f} | val {val_loss:.4f}')

        if val_loss < best_val - min_delta:
            best_val = val_loss
            patience_counter = 0
            torch.save(score_model.state_dict(), checkpoint_path)
            print(f'  -> saved best checkpoint (val={val_loss:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1} (no val improvement for {patience} epochs).')
                break

    np.savez(loss_path, train=np.array(train_losses), val=np.array(val_losses))
    print(f'Best val loss: {best_val:.4f}')
    return score_model, sde, train_losses, val_losses


## Generation of new samples

In [12]:
def setup_for_sampling(sde_params):
    beta_min, beta_max = sde_params
    params_str = f'{beta_min}_{beta_max}'
    checkpoint_path = os.path.join(
        checkpoint_dir, f'ckpt_fashionmnist_{n_epochs}epochs_{params_str}.pth'
    )
    sde = VPSDE(beta_min, beta_max, num_steps)
    score_model = ScoreNet(marginal_proba=sde.marginal_proba).to(device)
    score_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    score_model.eval()
    return sde, score_model


### Sampling

### Problem 5.C.ii–iii — Training

**Settings:** 50 epochs, LR=1e-4, batch=64, β=[0.01, 5.0], 1000 reverse steps.

**Early stopping:** save checkpoint on val-loss improvement (≥`min_delta`); stop after `patience` epochs without improvement.


In [ ]:
def plot_losses(train_losses, val_losses, sde_params):
    beta_min, beta_max = sde_params
    epochs = np.arange(1, len(train_losses) + 1)
    plt.figure(figsize=(8, 4))
    plt.semilogy(epochs, train_losses, label='train')
    if val_losses:
        plt.semilogy(epochs[:len(val_losses)], val_losses, label='val')
    plt.xlabel('Epoch')
    plt.ylabel('DSM loss (log scale)')
    plt.title(f'VP score model — β ∈ [{beta_min}, {beta_max}]')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(checkpoint_dir, f'losses_{beta_min}_{beta_max}.png')
    plt.savefig(path, dpi=150)
    plt.show()
    print(f'Saved: {path}')


In [ ]:
# Train (~10 min on A100). Set force_retrain=True to train from scratch.
score_model, sde, train_losses, val_losses = train(sde_params, force_retrain=False)
plot_losses(train_losses, val_losses, sde_params)


In [ ]:
# Problem 5.C.iii — 64 EM samples
score_model.eval()
em_samples = Euler_Maruyama_sampler(score_model, sde, n_samples, num_steps=num_steps, device=device)
plot_images(em_samples.clamp(0, 1))
plt.title(f'EM samples (N={num_steps}, β=[{beta_min}, {beta_max}])')
plt.savefig(os.path.join(checkpoint_dir, f'em_samples_{beta_min}_{beta_max}.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Problem 5.C.iv — PC sampler, 1 corrector step
pc_samples_1 = predictor_corrector_sampler(
    score_model, sde, n_samples, num_steps=num_steps, num_corrector_steps=1, device=device,
)
plot_images(pc_samples_1.clamp(0, 1))
plt.title(f'PC samples — 1 corrector step')
plt.savefig(os.path.join(checkpoint_dir, f'pc_samples_n1_{beta_min}_{beta_max}.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Problem 5.C.iv — PC sampler, 5 corrector steps
pc_samples_5 = predictor_corrector_sampler(
    score_model, sde, n_samples, num_steps=num_steps, num_corrector_steps=5, device=device,
)
plot_images(pc_samples_5.clamp(0, 1))
plt.title(f'PC samples — 5 corrector steps')
plt.savefig(os.path.join(checkpoint_dir, f'pc_samples_n5_{beta_min}_{beta_max}.png'), dpi=150, bbox_inches='tight')
plt.show()


# Part 4D: Diffusion models for inverse problems

## EC (10pts) : inpainting -- keep until you're done with the pset
This part will require you to read and understand [[Song22]](https://arxiv.org/pdf/2111.08005.pdf) Solving Inverse Problems In Medical Imaging With Score-Based Generative Models.

In our toy example, our measurement matrix A will be an explicit inpainting matrix that replaces some of the pixels in an image by 0 (blackening them out).

Your job will be to modify the sampling process to include conditioning on the perturbed measurements at various t.

**Important: do not attempt until you are entirely satisfied with your work for all of the rest of the assignment (Parts 1 through 5).**

**Important: Minimum guidance will be provided in OH or on Piazza. 🎶🎵You're on your own, kid. You always have been🎵🎶**

**Expectations:** The expected result is a plot with 3 rows: clean images, inpainted images, reconstructed images. The plot **must** include the PSNR of each reconstructed image. You should use at least 5 clean images. We wrote the plotting function for you. We also expect you to explain and justify your implementation choices. Everything (plot, explanations, justifications) **must** appear in your submission PDF.

In [ ]:
# helper function to inpaint with 0 pixels and get the subsampling matrix
def inpaint(images, ratio=0.05):
    num_pixels = images.shape[-2] * images.shape[-1]
    num_samples = int(ratio * num_pixels)
    # create subsampling matrix A
    A = torch.eye(num_pixels, device=images.device)
    for pixel in random.sample(range(0, num_pixels), num_samples):
        A[pixel][pixel] = 0
    # black out pixels in images using A (a binary matrix with zeroes where we want to black out pixels)
    inpainted_images = images.clone()
    for i in range(len(images)):
        inpainted_images[i] = torch.reshape(torch.matmul(A, inpainted_images[i].view(num_pixels)), images[0].shape)
    return inpainted_images, A

In [ ]:
# helper function to compute the peak signal-to-noise ratio (PSNR)
def psnr(clean, noisy):
    # our range of values is [0.,1.]
    eps = 1e-8
    # TODO: compute psnr
    psnr = None
    return psnr

In [ ]:
# helper function to plot samples
def plot_before_after(clean_images, imgs_before, imgs_after, title=""):
    assert(imgs_before.shape[0] == imgs_after.shape[0])
    fig, axs = plt.subplots(3, imgs_before.shape[0], figsize=(16, 10))
    # plot 3 rows: clean, then subsampled, then denoised
    for i, images in enumerate([clean_images, imgs_before, imgs_after]):
        for j, image in enumerate(images):
            axs[i][j].imshow(image.cpu().permute(1, 2, 0).squeeze())
            axs[i][j].set_xticks([])
            axs[i][j].set_yticks([])
    # compute PSNR
    for j, image in enumerate(imgs_after):
        clean = clean_images[j].cpu().permute(1,2,0).squeeze()
        noisy = image.cpu().permute(1,2,0).squeeze()
        psnr_val = psnr(clean, noisy).item()
        axs[2][j].set_title('PSNR: {:.3f}'.format(psnr_val), y=-0.2)
    fig.suptitle(title, size=20)

In [ ]:
# helper functions to condition the reverse diffusion
def get_y_t(images, t, marginal_proba):
    # vector of t
    ts = t * torch.ones(images.shape[0], device=images.device)
    ts = ts[:, None, None, None]

    # sample some noise
    z = torch.randn_like(images)

    # perturb at level t
    _, std = marginal_proba(x=0, t=t)
    perturbed_images = images + z * std
    return perturbed_images

def lbda_scheduler(t, lbda, param):
    param = torch.tensor(param)
    f_t = param*t
    lbda = lbda * f_t
    return lbda

def condition_on_inpainted_y(raw_images, x_t, t, marginal_prob_std, A, lbda=.01, lbda_param=10):
    y_t = get_y_t(raw_images, t, marginal_prob_std)
    lbda = lbda_scheduler(t, lbda, param=lbda_param)
    # TODO: YOUR CODE HERE
    P_inv, T = None, None
    # END OF YOUR CODE
    A = A
    # turn images into column vectors
    flat_y_t = torch.flatten(y_t, start_dim=1)
    flat_x_t = torch.flatten(x_t, start_dim=1)
    lbda = lbda[:, None]
    # x_prime is a weighted function of x and y
    y_influence = lbda * torch.matmul(A, torch.matmul(P_inv, flat_y_t.T)).T
    x_influence = (1 - lbda) * torch.matmul(A, torch.matmul(T, flat_x_t.T)).T + \
                  torch.matmul(torch.eye(A.shape[0], device=A.device) - A,
                               torch.matmul(T, flat_x_t.T)).T
    x_t_prime = torch.reshape(y_influence + x_influence, x_t.shape)
    return x_t_prime

In [ ]:
# Inpainted images
num_images = None
data, _ = next(iter(train_loader))
clean_images = data[:num_images].to(device)
corrupted_images, A = inpaint(clean_images, ratio=0.75)

# Denoised images
recovered_images = None

# Expected plot
plot_before_after(clean_images, corrupted_images, recovered_images, title="Inverting 75% inpainting on FashionMNIST")

# Part 5: Diffusion models on a larger scale

As mentioned in the write-up, you only need to include your code for plotting in this notebook for part 5. Here we provide some helper code if you run the experiments here in Google Colab.

### Helper code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install blobfile mpi4py

#### Change the directory (TODO)

In [ ]:
directory = '/content/drive/MyDrive/path/to/guided-diffusion/repo'

In [ ]:
!wget https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_classifier.pt -P {directory}/models
!wget https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt -P {directory}/models
%cd {directory}

In [ ]:
!mv scripts/classifier_sample.py classifier_sample.py

#### Example code for running a script

In [ ]:
import os
# specify directory to save samples
os.environ['OPENAI_LOGDIR'] = '/content/test'

In [ ]:
# define flags
SAMPLE_FLAGS = """
    --batch_size 1
    --num_samples 2
    --timestep_respacing 250
"""
MODEL_FLAGS = """
    --attention_resolutions 32,16,8
    --class_cond False
    --diffusion_steps 1000
    --image_size 256
    --learn_sigma True
    --noise_schedule linear
    --num_channels 256
    --num_head_channels 64
    --num_res_blocks 2
    --resblock_updown True
    --use_fp16 True
    --use_scale_shift_norm True
"""

# run a script
!python classifier_sample.py {MODEL_FLAGS.replace('\n', '')} \
    --classifier_scale 10.0 \
    --classifier_path models/256x256_classifier.pt \
    --model_path models/256x256_diffusion_uncond.pt \
    {SAMPLE_FLAGS.replace('\n', '')}

### A: unconditional generation

In [ ]:
# todo

### B: progression over time visualization

In [ ]:
# todo

### C: interpolation visualization

In [ ]:
# todo

### D: conditional generation

In [ ]:
# todo

### E: conditional generation with different classifier scales

In [ ]:
# todo